<a href="https://colab.research.google.com/github/Demosthene-OR/Student-AI-and-Data-Management/blob/main/90_Intro_to_Data_Science_with_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<img src="https://prof.totalenergies.com/wp-content/uploads/2024/09/TotalEnergies_TPA_picto_DegradeRouge_RVB-1024x1024.png" height="150" width="150">
<hr style="border-width:2px;border-color:#75DFC1">
<h1 style = "text-align:center" > Exercises Introduction to Data Science with Python </h1> 
<h2 style = "text-align:center"> Teacher: Olivier Renouard </h2> 
<hr style="border-width:2px;border-color:#75DFC1">

This notebook follows a simple learning path:

1. load and inspect data,
2. explore data quality issues,
3. build a first **regression** model,
4. compare several **classification** models.

Datasets used in this notebook:
- `ex02_Salary_Data_with_missing.csv`
- `CarPrice_for_EDA.csv`
- `telecom_churn_ready.csv`

Important note: the datasets must be stored in the **same folder** as this notebook.

## 0. Learning goals

By the end of this notebook, you should be able to:
- load a dataset with `pandas`,
- inspect its structure,
- detect missing values and outliers,
- prepare `X` and `y` variables,
- train a linear regression model,
- compare several classification models.

In [ ]:
# Import the main libraries used throughout the notebook
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Import scikit-learn tools for splitting data, training models, and evaluation
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Define display options and plotting style for cleaner outputs
sns.set_theme(style="whitegrid")
pd.set_option('display.max_columns', 200)

## 1. Mini example: Salary dataset

We start with a very simple dataset to understand the overall machine learning workflow:
- one input feature: `YearsExperience`,
- one target variable: `Salary`.

This file contains a few missing values on purpose so that we can practice data cleaning.

In [ ]:
# Load the salary dataset
# The file is expected to be in the same folder as the notebook
url = "https://raw.githubusercontent.com/Demosthene-OR/Student-AI-and-Data-Management/main/data_90/"
file = "ex02_Salary_Data_with_missing.csv" 
salary_df = pd.read_csv(url+file)

# Display the first rows
salary_df.head()

In [ ]:
# Display the shape, data types, and missing values


In [ ]:
# Plot the salary distribution and a boxplot to identify spread and possible outliers
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(salary_df['Salary'], kde=True, ax=axes[0])
axes[0].set_title('Salary distribution (with missing values)')
sns.boxplot(y=salary_df['Salary'], ax=axes[1])
axes[1].set_title('Salary boxplot')
plt.tight_layout()
plt.show()

### 1.1 Handling missing values

Here we create two common versions of the dataset:
- mean imputation,
- median imputation.

Later in this notebook, we will keep the median-imputed version because it is often more robust when outliers are present.

In [ ]:
# Create two copies of the dataset for two different imputation strategies
salary_mean = salary_df.copy()
salary_median = salary_df.copy()

# Replace missing salary values with the mean
# salary_mean['Salary'] = 

# Replace missing salary values with the median
# salary_median['Salary'] = 

# Check that missing values have been removed
pd.DataFrame({
    'missing_after_mean_imputation': salary_mean.isna().sum(),
    'missing_after_median_imputation': salary_median.isna().sum()
})

In [ ]:
# Compare the original distribution with the two imputed versions
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.histplot(salary_df['Salary'], kde=True, ax=axes[0])
axes[0].set_title('Original')
sns.histplot(salary_mean['Salary'], kde=True, ax=axes[1], color='orange')
axes[1].set_title('Mean imputation')
sns.histplot(salary_median['Salary'], kde=True, ax=axes[2], color='green')
axes[2].set_title('Median imputation')
plt.tight_layout()
plt.show()

## 2. Main dataset: CarPrice

We now move to a richer dataset that is closer to a real-world use case.
The goal is to learn a basic exploratory data analysis workflow before modeling.

In [ ]:
# Load the car price dataset prepared for EDA
# The file is expected to be in the same folder as the notebook
url = "https://raw.githubusercontent.com/Demosthene-OR/Student-AI-and-Data-Management/main/data_90/"
file = "CarPrice_for_EDA.csv" 
car_df = pd.read_csv(url+file)

# Display the first rows
car_df.head()

In [ ]:
# Inspect the dataset structure (print the shape, types & missing values)


In [ ]:
# Show descriptive statistics for the dataset


### 2.1 Structure summary with `df.info()` and `df.describe()`

Two very useful commands at the beginning of any analysis are:
- `df.info()` to inspect the structure of the dataset,
- `df.describe()` to summarize numerical variables.

In [ ]:
# Use .info() to inspect the CarPrice dataset structure
car_df.info()

In [ ]:
# Use .describe() to summarize numerical columns
car_df.describe()

### 2.2 Mean, median, quartiles, and IQR

Reminder:
- mean = arithmetic average,
- median = middle value,
- Q1 = 25th percentile,
- Q3 = 75th percentile,
- IQR = Q3 - Q1.

The IQR is often used to detect outliers.

In [ ]:
# TODO: compute the mean, median, Q1, Q3, and IQR for the CarPrice price variable
# Hint: use .mean(), .median(), and .quantile()

### 2.3 Visualizing a normal distribution

This plot is only meant as a visual reference for what a bell-shaped distribution looks like.

In [ ]:
# Generate a sample from a normal distribution and visualize it
np.random.seed(0)
sample_normal = np.random.normal(loc=0, scale=1, size=1000)

plt.figure(figsize=(8, 4))
sns.histplot(sample_normal, kde=True, bins=30)
plt.title('Example of a normal distribution')
plt.xlabel('Value')
plt.ylabel('Frequency')
plt.show()

### 2.4 QQ plot for normality checking

A QQ plot helps us compare the distribution of a variable with a theoretical normal distribution.

In [ ]:
# Import scipy for the QQ plot
from scipy import stats

In [ ]:
# TODO: create a QQ plot for the CarPrice price variable
# Hint: use stats.probplot(..., dist='norm', plot=plt)

### 2.1 Numerical variables

In [ ]:
# Select all numerical columns
num_cols = car_df.select_dtypes(include=np.number).columns.tolist()
print(num_cols)

In [ ]:
# Select a few important numerical columns for visualization
selected_cols = [c for c in ['price', 'horsepower', 'enginesize', 'curbweight', 'highwaympg'] if c in car_df.columns]

# Plot histograms for these variables


In [ ]:
# Plot a correlation heatmap for selected variables


### 2.2 Outliers

We use boxplots to visually identify extreme values.

In [ ]:
# Draw boxplots to detect potential outliers
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.flatten()
for ax, col in zip(axes, selected_cols[:4]):
    # display boxplot 
    # display a titel above
plt.tight_layout()
plt.show()

### 2.3 Minimal cleaning for modeling

In this notebook, we create a simple cleaned version by:
- removing rows where the target `price` is missing,
- imputing the median for a few useful numerical features.

In [ ]:
# Copy the dataset before cleaning
car_model_df = car_df.copy()

# Remove rows where the target variable is missing (use argument subset=['price'] in dropna())


# Fill missing values in selected numerical features with the median


# Verify that the selected modeling columns no longer contain missing values


## 3. Simple linear regression on Salary

In [ ]:
# TODO: split the Salary dataset into train and test sets using train_test_split
# Hint: use test_size=0.25 and random_state=0

# Example solution (uncomment to reveal):
# X_train_s, X_test_s, y_train_s, y_test_s = train_test_split(
#     X_salary, y_salary, test_size=0.25, random_state=0
# )
# print("X_train shape:", X_train_s.shape)
# print("X_test shape:", X_test_s.shape)
# print("y_train shape:", y_train_s.shape)
# print("y_test shape:", y_test_s.shape)


In [ ]:
# TODO: fit a LinearRegression model on the Salary training data
# Hint: create the model and call .fit(X_train_s, y_train_s)

# Example solution (uncomment to run):
# lin_salary = LinearRegression()
# lin_salary.fit(X_train_s, y_train_s)
# y_pred_s = lin_salary.predict(X_test_s)
# print("Intercept:", lin_salary.intercept_)
# print("Coefficient:", lin_salary.coef_[0])
# print("MAE:", mean_absolute_error(y_test_s, y_pred_s))
# print("RMSE:", np.sqrt(mean_squared_error(y_test_s, y_pred_s)))
# print("R2:", r2_score(y_test_s, y_pred_s))


In [ ]:
# TODO: visualize the training data and regression line for Salary
# Hint: use plt.scatter for data points and plt.plot for the regression line


In [ ]:
# Visualize the test set and predicted values
plt.figure(figsize=(8, 5))
plt.scatter(X_test_s, y_test_s, color='orange', label='Test data')
plt.plot(X_test_s, y_pred_s, 'bo', label='Predicted values')
plt.title('Salary vs Experience - Test set')
plt.xlabel('YearsExperience')
plt.ylabel('Salary')
plt.legend()
plt.show()

## 4. Linear regression on CarPrice

We now predict `price` using several numerical features.
This section illustrates that real-world datasets usually require more preparation than toy examples.

In [ ]:
# Select a few numerical features for the regression model
features = [c for c in ['horsepower', 'enginesize', 'curbweight', 'highwaympg'] if c in car_model_df.columns]
X_car = car_model_df[features]
y_car = car_model_df['price']

# Display the selected features and dataset dimensions
print('Features used:', features)
print('X shape:', X_car.shape)
print('y shape:', y_car.shape)

In [ ]:
# TODO: train a LinearRegression model on the CarPrice dataset
# Hint: use X_train_c, y_train_c and then predict on X_test_c

# Example solution (uncomment to run):
# lin_car = LinearRegression()
# lin_car.fit(X_train_c, y_train_c)
# y_pred_c = lin_car.predict(X_test_c)
# print("MAE:", mean_absolute_error(y_test_c, y_pred_c))
# print("RMSE:", np.sqrt(mean_squared_error(y_test_c, y_pred_c)))
# print("R2:", r2_score(y_test_c, y_pred_c))


In [ ]:
# TODO: build a DataFrame of regression coefficients for CarPrice
# Hint: combine the feature names and lin_car.coef_ into a DataFrame


In [ ]:
# Plot real prices versus predicted prices
plt.figure(figsize=(7, 5))
plt.scatter(y_test_c, y_pred_c, alpha=0.7)
plt.xlabel('Actual price')
plt.ylabel('Predicted price')
plt.title('CarPrice - Actual vs Predicted')
min_val = min(y_test_c.min(), y_pred_c.min())
max_val = max(y_test_c.max(), y_pred_c.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--')
plt.show()

## 5. Classification: telecom churn

We now switch to another type of machine learning problem.
The target variable `Churn` is binary:
- `1` means the customer leaves,
- `0` means the customer stays.

In [ ]:
# Load the telecom churn dataset
# The file is expected to be in the same folder as the notebook
url = "https://raw.githubusercontent.com/Demosthene-OR/Student-AI-and-Data-Management/main/data_90/"
file = "telecom_churn_ready.csv" 
telecom_df = pd.read_csv(url+file)

# Display the first rows
telecom_df.head()

In [ ]:
# Inspect shape, columns, and total number of missing values


In [ ]:
# Compute the class distribution of the target variable


In [ ]:
# Visualize the percentage of each class


### 5.1 Building X and y

In [ ]:
# Define X as all input variables except the last column ('Churn')
# Define y as the target column 'Churn'


# Display dimensions and a preview of X


In [ ]:
# Split the data into train and test sets
# stratify=y keeps the same class balance in both subsets


### 5.2 Comparing several models

In [ ]:
# TODO: compare several classification models using cross-validation
# Hint: loop over the models dictionary, call cross_validate, and collect metrics


In [ ]:
# TODO: plot the F1-score for each model
# Hint: use sns.barplot with scores_df.index and scores_df["F1-score"]


## 6. Conclusion

In this notebook, we learned how to:
- inspect and clean a dataset,
- prepare `X` and `y`,
- train a linear regression model,
- compare several classification models.

Natural next steps would be:
- encoding categorical variables,
- scaling features,
- using more advanced validation strategies,
- interpreting models in greater detail.